# Prediciendo la diabetes con Random Forest

Proyecto bootcamp (EDA completo + RandomForestClassifier + análisis de hiperparámetros + guardado del modelo).


In [ ]:
# =========================
# Paso 1: Carga del dataset
# =========================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# (Opcional) mejor estética de gráficos
sns.set_theme(style="whitegrid")

# Cargamos el dataset desde el link del bootcamp
url = "https://breathecode.herokuapp.com/asset/internal-link?id=930&path=diabetes.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()


In [ ]:
# ==================================
# Paso 2: Exploración inicial (EDA)
# ==================================

# Información general de columnas y tipos
df.info()

# Estadísticas descriptivas
df.describe()


In [ ]:
# --------------------------
# Duplicados y nulos
# --------------------------

print("Duplicados:", df.duplicated().sum())
print("\nNulos por columna:\n", df.isnull().sum())

# Distribución de clases (balance)
print("\nBalance de clases (Outcome):\n", df["Outcome"].value_counts())
print("\nProporción de clases:\n", df["Outcome"].value_counts(normalize=True))


In [ ]:
# --------------------------
# Distribución de la variable objetivo
# --------------------------

plt.figure(figsize=(5,4))
sns.countplot(x="Outcome", data=df)
plt.title("Distribución de Outcome (0 = no diabetes, 1 = diabetes)")
plt.xlabel("Outcome")
plt.ylabel("Conteo")
plt.tight_layout()
plt.show()


In [ ]:
# --------------------------
# Univariante (numéricas) en layout
# --------------------------

# Todas las columnas son numéricas en este dataset
num_cols = [c for c in df.columns if c != "Outcome"]

n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=20, kde=True, ax=axes[i])
    axes[i].set_title(f"Histograma: {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frecuencia")

# Apagamos ejes sobrantes
for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

# Boxplots (outliers)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(f"Boxplot: {col}")
    axes[i].set_xlabel(col)

for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# --------------------------
# Bivariante: numéricas vs Outcome (layout)
# --------------------------

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x="Outcome", y=col, data=df, ax=axes[i])
    axes[i].set_title(f"{col} vs Outcome")
    axes[i].set_xlabel("Outcome")
    axes[i].set_ylabel(col)

for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# --------------------------
# Correlaciones
# --------------------------

corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Limpieza: ceros "imposibles" -> NaN -> mediana
# ============================================

# En este dataset, 0 suele significar "no medido" en estas columnas
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# Conteo de ceros antes del reemplazo
for col in zero_as_missing:
    print(f"{col}: ceros = {(df[col] == 0).sum()}")

# Reemplazo 0 por NaN en las columnas indicadas
df_clean = df.copy()
df_clean[zero_as_missing] = df_clean[zero_as_missing].replace(0, np.nan)

print("\nNulos después de reemplazar 0->NaN:\n", df_clean.isnull().sum())


In [ ]:
# --------------------------
# Imputación por mediana
# --------------------------

from sklearn.impute import SimpleImputer

# Separamos X e y
X = df_clean.drop(columns=["Outcome"])
y = df_clean["Outcome"]

# Imputamos (solo numéricas) usando la mediana
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Verificación rápida
X_imputed.isnull().sum()


In [ ]:
# ============================================
# Split: Train / Val / Test (estratificado)
# ============================================

from sklearn.model_selection import train_test_split

# 1) separamos test (20%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2) separamos validación desde trainval (20% de trainval -> 16% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape, "y_val  :", y_val.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)

print("\nProporción clase 1 (diabetes):")
print("train:", y_train.mean(), "val:", y_val.mean(), "test:", y_test.mean())


In [ ]:
# ============================================
# Paso 2: Random Forest (modelo base)
# ============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay
)

# Modelo base (por defecto n_estimators=100)
rf_base = RandomForestClassifier(
    random_state=42
)

# Entrenamiento
rf_base.fit(X_train, y_train)

# Predicción en validación
y_val_pred = rf_base.predict(X_val)
y_val_proba = rf_base.predict_proba(X_val)[:, 1]

# Métricas en validación
acc  = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred, zero_division=0)
rec  = recall_score(y_val, y_val_pred, zero_division=0)
f1   = f1_score(y_val, y_val_pred, zero_division=0)
auc  = roc_auc_score(y_val, y_val_proba)

print("Random Forest BASE (Validación)")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred)
plt.title("Matriz de confusión (Validación) - RF Base")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Paso 2: Probar hiperparámetros (n_estimators y max_depth)
# ============================================

# Probamos varios números de árboles
n_estimators_list = [10, 50, 100, 200, 300, 500]

# Probamos varias profundidades (None = sin límite)
max_depth_list = [None, 2, 3, 4, 5, 7, 10]

# Guardamos accuracies para graficar
acc_by_n = []
acc_by_depth = []

# 1) Variar n_estimators (manteniendo max_depth fijo)
for n in n_estimators_list:
    model = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    acc_by_n.append(accuracy_score(y_val, pred))

# 2) Variar max_depth (manteniendo n_estimators fijo)
for d in max_depth_list:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=d,
        random_state=42
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    acc_by_depth.append(accuracy_score(y_val, pred))

# Gráficos en layout 1x2 (para ahorrar espacio)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(n_estimators_list, acc_by_n, marker="o")
axes[0].set_title("Accuracy vs n_estimators (Validación)")
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True)

axes[1].plot([str(d) for d in max_depth_list], acc_by_depth, marker="o")
axes[1].set_title("Accuracy vs max_depth (Validación)")
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Elegir el mejor modelo según validación (simple)
# ============================================

# Buscamos el mejor n_estimators
best_n = n_estimators_list[int(np.argmax(acc_by_n))]
best_acc_n = float(np.max(acc_by_n))

# Buscamos el mejor max_depth
best_d = max_depth_list[int(np.argmax(acc_by_depth))]
best_acc_d = float(np.max(acc_by_depth))

print("Mejor n_estimators según validación:", best_n, "Accuracy:", best_acc_n)
print("Mejor max_depth según validación:", best_d, "Accuracy:", best_acc_d)

# Construimos un modelo final combinando ambos "mejores" (heurística simple)
rf_final = RandomForestClassifier(
    n_estimators=best_n,
    max_depth=best_d,
    random_state=42
)

# Reentrenamos usando train + val (más datos) antes de evaluar en test
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

rf_final.fit(X_train_final, y_train_final)

# Evaluación final en test
y_test_pred = rf_final.predict(X_test)
y_test_proba = rf_final.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred, zero_division=0)
rec  = recall_score(y_test, y_test_pred, zero_division=0)
f1   = f1_score(y_test, y_test_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_test_proba)

print("\nRandom Forest FINAL (Test)")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred)
plt.title("Matriz de confusión (Test) - RF Final")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Paso 3: Guardar el modelo
# ============================================

from pickle import dump

dump(rf_final, open("random_forest_diabetes_42.sav", "wb"))

print("Modelo guardado como: random_forest_diabetes_42.sav")
